In [2]:
# =========================
# MNIST NN (CSV) - Assignment 2 Question 7
# 3-layer vs 4-layer NN, 80/20 split, >=100 epochs
# Framework: PyTorch
# =========================

### Cell 1 : Imports

In [3]:
# Cell 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


### Cell 2: Load MINIST Dataset

In [4]:
# Cell 2: Load dataset
csv_path = "MINST.csv"  # keep file in same folder as notebook

df = pd.read_csv(csv_path)

# first column = label
y = df["label"].values.astype(np.int64)
X = df.drop(columns=["label"]).values.astype(np.float32)

# normalize pixels
X = X / 255.0

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'MINST.csv'

### Cell 3: Train/Test split (80/20)

In [ ]:
# Cell 3: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
Xtr = torch.from_numpy(X_train)
ytr = torch.from_numpy(y_train)
Xte = torch.from_numpy(X_test)
yte = torch.from_numpy(y_test)

train_ds = TensorDataset(Xtr, ytr)
test_ds = TensorDataset(Xte, yte)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))


### Cell 4: Define 3 - Layer Neural Network

In [ ]:
# Cell 5: 4-layer NN (input → hidden1 → hidden2 → output)
class MLP4(nn.Module):
    def __init__(self, in_dim=784, h1=256, h2=128, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, out_dim)
        )
    def forward(self, x):
        return self.net(x)


### Cell 5: Define 4- Layer Neural Network

In [ ]:
# Cell 5: 4-layer NN (input → hidden1 → hidden2 → output)
class MLP4(nn.Module):
    def __init__(self, in_dim=784, h1=256, h2=128, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, out_dim)
        )
    def forward(self, x):
        return self.net(x)


### Cell 6: Training Function

In [ ]:
# Cell 6: Train + evaluate function
def train_model(model, epochs=100, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_loss_hist, test_loss_hist = [], []
    train_acc_hist, test_acc_hist = [], []

    for epoch in range(epochs):

        # ----- training -----
        model.train()
        total_loss, correct, total = 0, 0, 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (logits.argmax(1) == yb).sum().item()
            total += yb.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total

        # ----- testing -----
        model.eval()
        total_loss, correct, total = 0, 0, 0

        with torch.no_grad():
            for xb, yb in test_loader:
                logits = model(xb)
                loss = criterion(logits, yb)

                total_loss += loss.item()
                correct += (logits.argmax(1) == yb).sum().item()
                total += yb.size(0)

        test_loss = total_loss / len(test_loader)
        test_acc = correct / total

        train_loss_hist.append(train_loss)
        test_loss_hist.append(test_loss)
        train_acc_hist.append(train_acc)
        test_acc_hist.append(test_acc)

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Train Acc {train_acc:.4f} | Test Acc {test_acc:.4f}")

    return train_loss_hist, test_loss_hist, train_acc_hist, test_acc_hist


### Cell 7: Training 3 - Layer Network

In [ ]:
# Cell 7: Train 3-layer NN
torch.manual_seed(42)

model3 = MLP3()
hist3 = train_model(model3, epochs=100)

print("Final Test Accuracy (3-layer):", hist3[3][-1])


### Cell 8: Training 4 - Layer Network

In [ ]:
# Cell 8: Train 4-layer NN
torch.manual_seed(42)

model4 = MLP4()
hist4 = train_model(model4, epochs=100)

print("Final Test Accuracy (4-layer):", hist4[3][-1])


### Cell 9: Plot Training VS Testing Error (3-Layer)

In [ ]:
# Cell 9: Plot 3-layer loss
plt.figure()
plt.plot(hist3[0], label="Train Loss")
plt.plot(hist3[1], label="Test Loss")
plt.title("3-layer NN Error vs Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


### Cell 10: Plot Training Vs Testing Error (4-Layer)

In [ ]:
# Cell 10: Plot 4-layer loss
plt.figure()
plt.plot(hist4[0], label="Train Loss")
plt.plot(hist4[1], label="Test Loss")
plt.title("4-layer NN Error vs Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


### Cell 11: Softmax Example

In [ ]:
# Cell 11: Softmax probabilities example
model3.eval()

with torch.no_grad():
    sample_logits = model3(Xte[:5])
    probs = torch.softmax(sample_logits, dim=1)

print("Softmax probabilities:")
print(probs)
